### Tesouro Direto EDA

Bonds:
- ntnb: Tesouro IPCA+. Input for real yeald curve
- ntnf: Tesouro Prefixado2. Pays nominal coupon. Long end nominal curve
- ltn: Tesouro Prefixdo. Zero coupon. short-to-medium nonimal curve

*Goal*: confirm parsing, inspect yield term structures, check gaps or anomalies, understanding yield curves before bootstrapping

In [ ]:
import sys
sys.path.insert(0, '..')

from datetime import date, timedelta
from src.ingestion.tesouro import fetch_all

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.cm as cm
import numpy as np

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

#### Fetching

In [ ]:
END = date.today()
START = END - timedelta(days=365 * 5) # Five years
data = fetch_all(start_date=START, end_date=END)

for name, df in data.items():
    n_invalid = df[~df['is_valid']].sum()
    n_mat = df['maturity'].nunique()
    print(f"{name}: {len(df):6d} rows, {n_invalid} invalid rows, {n_mat} maturities")

### Schema and sample

In [ ]:
for name, df in data.items():
    print(f"\n{name}")
    display(df[['date', 'bond_name', 'maturity', 'pu_base', 'rate_annual', 'is_valid']].head(4))

### Available maturities per bond type

In [ ]:
for name, df in data.items():
    maturities = df['maturity'].drop_duplicates().sort_values()
    today = pd.Timestamp(END)
    ttm_years = ((maturities - today).dt.days / 365).round(1)
    print(f"\n {name}: {len(maturities)} maturities")
    for mat, ttm in zip(maturities, ttm_years):
        label = f"  {mat.date()}  ({ttm}y)"
        if ttm < 0:
            label += "expired"
        print(label)

###  Descriptive statistics per bond type

In [ ]:
for name, df in data.items():
    print(f"{name}:rate_annual")
    display(
        df[df['is_valid']]
        .groupby('bond_name')['rate_annual']
        .describe()
        .round(4)
    )